# ATC Multi-Agent GRPO Training (with ADAPT)

**Runtime**: `Runtime → Change runtime type → GPU (T4 or better)`

**Default model**: `Qwen/Qwen2.5-1.5B-Instruct` — safe on T4 (16 GB).  
Change `MODEL_NAME` to `Qwen/Qwen2.5-7B-Instruct` for L4 / A100.

### What this notebook does
1. Installs dependencies (correct order — avoids `retry` import error)
2. Clones the ATS repo (`cleaned` branch)
3. Verifies ADAPT structural-inference imports and reward diversity
4. Trains **AMAN / DMAN / GENERATOR / SUPERVISOR / ADAPT** via GRPO
5. Plots per-role reward curves including ADAPT domain-transfer rewards
6. Compares trained vs heuristic baseline (including ADAPT metrics)
7. Saves all plots as `.png`

## 1. Configure

In [ ]:
import os, sys
from pathlib import Path

IS_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
IS_KAGGLE = os.path.exists('/kaggle/working')
PLATFORM  = 'colab' if IS_COLAB else ('kaggle' if IS_KAGGLE else 'local')
print(f'Platform: {PLATFORM}')

# ── Repository ────────────────────────────────────────────────────────────────
REPO_URL    = 'https://github.com/yvishi/ats.git'
REPO_BRANCH = 'cleaned'

# ── Model ─────────────────────────────────────────────────────────────────────
# T4 (16 GB):  Qwen/Qwen2.5-1.5B-Instruct  ← default, safe
# T4 (push):   Qwen/Qwen2.5-3B-Instruct    ← set N_GENERATIONS=2
# L4 / A100:   Qwen/Qwen2.5-7B-Instruct
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

# ── Training ──────────────────────────────────────────────────────────────────
EPISODES             = 5      # 150 eps ≈ 1-2 h on T4 with 1.5B
N_GENERATIONS        = 4      # GRPO group size — use 2 if OOM
LORA_RANK            = 16
SEED                 = 42
DOMAIN_EPISODE_RATIO = 0.30   # fraction of episodes sent to ADAPT agent

# ── W&B (optional) ────────────────────────────────────────────────────────────
WANDB_API_KEY = ''            # paste key, or use Colab/Kaggle secrets
WANDB_PROJECT = 'atc-multiagent-grpo'

# ── Paths ─────────────────────────────────────────────────────────────────────
if IS_COLAB:
    REPO_DIR   = Path('/content/ats')
    OUTPUT_DIR = Path('/content/drive/MyDrive/atc-outputs')
elif IS_KAGGLE:
    REPO_DIR   = Path('/kaggle/working/ats')
    OUTPUT_DIR = Path('/kaggle/working/atc-outputs')
else:
    REPO_DIR   = Path('./ats')
    OUTPUT_DIR = Path('./atc-outputs')

PLOTS_DIR = OUTPUT_DIR / 'plots'

print(f'Model:               {MODEL_NAME}')
print(f'Episodes:            {EPISODES}')
print(f'Group size:          {N_GENERATIONS}')
print(f'Domain episode ratio:{DOMAIN_EPISODE_RATIO} (ADAPT agent)')
print(f'Output:              {OUTPUT_DIR}')

## 2. Mount storage

In [ ]:
if IS_COLAB:
    USE_DRIVE = True
    if USE_DRIVE:
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        OUTPUT_DIR = Path('/content/atc-outputs')
        PLOTS_DIR  = OUTPUT_DIR / 'plots'
elif IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        if not WANDB_API_KEY:
            WANDB_API_KEY = _s.get_secret('WANDB_API_KEY')
        print('Loaded WANDB_API_KEY from Kaggle secrets')
    except Exception:
        pass

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output ready: {OUTPUT_DIR}')

## 3. Install dependencies

**Install order matters.**  
`unsloth` is installed **first** (with deps) so pip pins the correct `transformers` version.  
Installing `--no-deps` first causes `ImportError: cannot import name 'retry' from transformers.utils.generic`.

In [ ]:
import subprocess, sys

# Remove any pre-installed trl that may conflict
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'trl'], check=False)

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
        # Core training stack (unsloth first — pins compatible transformers)
        'unsloth[colab-new]',
        'trl==0.15.2',
        'transformers==4.51.3',
        'accelerate>=0.32.0',
        'peft>=0.12.0',
        'bitsandbytes>=0.43.0',
        # Data / utilities
        'datasets>=2.20.0',
        'numpy>=1.26.0',
        'matplotlib>=3.9.0',
        # Environment / API
        'openenv-core[core]>=0.2.3',
        'openai>=1.30.0',
        'fastapi>=0.111.0',
        'pydantic>=2.7.0',
        'uvicorn>=0.29.0',
    ],
    check=True,
)

os.environ['WANDB_MODE']             = 'disabled'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Install complete.')

## 4. Clone repository

In [ ]:
import shutil, os, sys, subprocess
from pathlib import Path

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--branch', REPO_BRANCH, '--depth', '1',
     REPO_URL, str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f'Working directory: {Path.cwd()}')

# Clear stale __pycache__ so Unsloth rebuilds compiled kernels fresh
for d in REPO_DIR.rglob('__pycache__'):
    shutil.rmtree(d, ignore_errors=True)
print('Cleared __pycache__')

## 5. Verify environment

In [ ]:
import torch

print(f'PyTorch:      {torch.__version__}')
print(f'CUDA:         {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:          {torch.cuda.get_device_name(0)}')
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM:         {vram_gb:.1f} GB')
    if vram_gb < 14:
        print('WARNING: <14 GB VRAM detected. Use 1.5B model + N_GENERATIONS=2')

import transformers, unsloth, trl, peft
print(f'Transformers: {transformers.__version__}')
print(f'Unsloth:      {unsloth.__version__}')
print(f'TRL:          {trl.__version__}')
print(f'PEFT:         {peft.__version__}')

# Verify the critical import that caused the original error
try:
    from transformers.utils.generic import retry  # noqa: F401
    print('transformers.utils.generic.retry: OK')
except ImportError:
    print('WARNING: retry not in transformers.utils.generic — re-run Cell 3')

from trl import GRPOConfig, GRPOTrainer
from unsloth import FastLanguageModel
print('GRPOTrainer + FastLanguageModel: OK')

# ── Dataset sanity check (all 5 roles including ADAPT) ────────────────────────
from training.dataset import build_episode_dataset
_check = build_episode_dataset(n_episodes=4, seed=0, include_adapt=True, domain_episode_ratio=0.30)
roles = sorted({r['agent_role'] for r in _check})
print(f'Dataset builder: OK — {len(_check)} samples, roles={roles}')
assert 'ADAPT' in roles, 'ADAPT role missing from dataset — check domain_episode_ratio > 0'
print('ADAPT role confirmed in dataset ✓')

# ── ADAPT structural-inference imports ────────────────────────────────────────
from multi_agent.adapt import (
    build_adapt_observation,
    apply_adapt_mapping,
    parse_adapt_action,
    _build_adapt_heuristic,
)
print('multi_agent.adapt: OK')

from training.reward_functions import adapt_reward_fn
print('adapt_reward_fn: OK')

# ── Domain catalog sanity check ───────────────────────────────────────────────
from domains.icu import get_icu_tasks
icu_tasks = get_icu_tasks()
print(f'ICU domain tasks: {len(icu_tasks)} tasks loaded')

## 6. Verify reward diversity

GRPO requires **non-zero reward variance** within each group of completions.  
Constant rewards → zero advantage → no gradient.

We verify all 5 roles (AMAN, DMAN, GENERATOR, SUPERVISOR, ADAPT).

In [ ]:
import statistics
from training.reward_functions import (
    aman_reward_fn, dman_reward_fn, adapt_reward_fn, _parse_partial_credit
)
from tasks import task_catalog
from domains.icu import get_icu_tasks
from multi_agent.adapt import build_adapt_observation, _build_adapt_heuristic
import json

catalog   = task_catalog()
atc_tid   = list(catalog.keys())[0]
atc_task  = catalog[atc_tid]
print(f'ATC test task: {atc_tid} ({len(atc_task.flights)} flights)')

# ── Check 1: AMAN reward diversity ────────────────────────────────────────────
aman_group = [
    'no json here at all',
    '{"arrival_slots": [{"flight_id": "AI101", "runway": "28L", "assigned_minute": 30, "hold_minutes": 0}], "rationale": "ok"}',
    '{"arrival_slots": [], "rationale": "runway unavailable"}',
    '{flight_id: broken json',
]
aman_rewards = aman_reward_fn(aman_group, task_id=[atc_tid]*4)
aman_std = statistics.stdev(aman_rewards)
print(f'AMAN rewards: {[round(r,4) for r in aman_rewards]}  std={aman_std:.4f}')
assert aman_std > 0, 'FAIL: AMAN rewards constant — zero GRPO gradient'
print('  PASS: AMAN reward diversity ✓')

# ── Check 2: ADAPT reward diversity ───────────────────────────────────────────
icu_tasks  = get_icu_tasks()
icu_tid    = list(icu_tasks.keys())[0]
icu_task   = icu_tasks[icu_tid]
print(f'\nICU test task: {icu_tid} ({len(icu_task.flights)} patients)')

obs = build_adapt_observation(icu_task, 'SAFETY_STRICT', domain_name='ICU')

# Heuristic baseline action
heuristic_action = _build_adapt_heuristic(obs, icu_task)
heuristic_json   = json.dumps(heuristic_action.model_dump())

adapt_group = [
    'no json here at all',
    heuristic_json,
    '{"entity_wake_map": {}, "entity_priority_map": {}, "rationale": "passthrough"}',
    '{entity_wake_map: broken}',
]
domain_task_json = json.dumps(icu_task.model_dump() if hasattr(icu_task, 'model_dump') else icu_task.__dict__)
adapt_rewards = adapt_reward_fn(
    adapt_group,
    task_id=[icu_tid]*4,
    domain_task_json=[domain_task_json]*4,
    supervisor_profile=['SAFETY_STRICT']*4,
)
adapt_std = statistics.stdev(adapt_rewards)
print(f'ADAPT rewards: {[round(r,4) for r in adapt_rewards]}  std={adapt_std:.4f}')
assert adapt_std > 0, 'FAIL: ADAPT rewards constant — check adapt_reward_fn'
print('  PASS: ADAPT reward diversity ✓')

# ── Check 3: _parse_partial_credit is monotonically generous ─────────────────
samples = [
    ('empty',              ''),
    ('plain text',         'I cannot do this'),
    ('has brace',          '{some content}'),
    ('has arrival_slots',  '{"arrival_slots": [...]}'),
    ('has flight+runway',  '{"arrival_slots": [{"flight_id":"X","runway":"28L"}]}'),
]
print('\n_parse_partial_credit monotonicity:')
prev = -1
for label, text in samples:
    c = _parse_partial_credit(text)
    print(f'  {label:25s}: {c:.4f}')
    assert c >= prev, f'Expected non-decreasing credit, got {c} < {prev}'
    prev = c

print('\nAll reward diversity checks PASSED.')

## 7. W&B setup (optional)

In [ ]:
import os

if IS_COLAB and not WANDB_API_KEY:
    try:
        from google.colab import userdata
        WANDB_API_KEY = userdata.get('WANDB_API_KEY')
        print('Loaded WANDB_API_KEY from Colab secrets')
    except Exception:
        pass

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY, relogin=True)
    os.environ['WANDB_PROJECT'] = WANDB_PROJECT
    os.environ['WANDB_MODE']    = 'online'
    print(f'W&B: online  project={WANDB_PROJECT}')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B: disabled (set WANDB_API_KEY to enable)')

## 8. Train all agents (AMAN / DMAN / GENERATOR / SUPERVISOR / ADAPT)

`training/train_grpo.py` runs:
- Load 4-bit QLoRA model
- Build episode dataset (30% of episodes routed to **ADAPT** domain-transfer task)
- GRPO loop over all 5 agent roles
- Save checkpoint + reward curves

> **OOM?** Set `N_GENERATIONS=2` or switch to `Qwen2.5-1.5B-Instruct`.

In [ ]:
import time, subprocess, sys, os, re
from collections import defaultdict

train_cmd = [
    sys.executable, 'training/train_grpo.py',
    '--model',         MODEL_NAME,
    '--output_dir',    str(OUTPUT_DIR),
    '--episodes',      str(EPISODES),
    '--n_generations', str(N_GENERATIONS),
    '--lora_rank',     str(LORA_RANK),
    '--seed',          str(SEED),
]

# ── Live reward tracking ───────────────────────────────────────────────────────
# TRL logs every logging_steps in dict format:
#   {'loss': 0.012, 'reward': 0.23, 'reward/AMAN': 0.31, 'reward/ADAPT': 0.18, 'step': 10, ...}
ALL_ROLES      = ('AMAN', 'DMAN', 'GENERATOR', 'SUPERVISOR', 'ADAPT', 'composite')
reward_history = defaultdict(list)
loss_history   = []
step_history   = []
last_step      = -1

_re_reward = re.compile(r"'reward(?:/(\w+))?':\s*([-\d.eE+]+)")
_re_loss   = re.compile(r"'loss':\s*([-\d.eE+]+)")
_re_step   = re.compile(r"'step':\s*(\d+)")

def _bar(val, lo=-1.0, hi=1.0, width=20):
    frac   = max(0.0, min(1.0, (val - lo) / (hi - lo)))
    filled = int(frac * width)
    return f'[{"#" * filled}{"-" * (width - filled)}] {val:+.3f}'

def _live_summary(step):
    elapsed = time.time() - t0
    print(f'\n── step {step:4d}  elapsed {elapsed/60:.1f} min ────────────────────')
    if loss_history:
        print(f'  loss        : {loss_history[-1]:.5f}')
    for role in ALL_ROLES:
        vals = reward_history.get(role, [])
        if not vals:
            continue
        recent = vals[-min(20, len(vals)):]
        mean_r = sum(recent) / len(recent)
        trend  = ''
        if len(vals) >= 40:
            half  = len(vals) // 2
            early = sum(vals[:half]) / half
            late  = sum(vals[half:]) / (len(vals) - half)
            trend = '  ↑' if late > early + 0.03 else ('  ↓' if late < early - 0.03 else '  →')
        tag = ' ◆' if role == 'ADAPT' else '  '
        print(f'  {role:10s}{tag}: {_bar(mean_r)}{trend}')
    print()

# ── Launch with live stdout streaming ─────────────────────────────────────────
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

print('Command:', ' '.join(train_cmd))
print('=' * 60)
t0   = time.time()
proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

for raw_line in proc.stdout:
    line = raw_line.rstrip('\n')
    print(line, flush=True)

    if "'reward'" in line or "'loss'" in line:
        m_step = _re_step.search(line)
        m_loss = _re_loss.search(line)
        if m_loss:
            loss_history.append(float(m_loss.group(1)))
        for m in _re_reward.finditer(line):
            role = m.group(1) or 'composite'
            reward_history[role].append(float(m.group(2)))
        if m_step:
            s = int(m_step.group(1))
            if s != last_step:
                last_step = s
                step_history.append(s)
                _live_summary(s)

proc.wait()
elapsed = time.time() - t0
print('=' * 60)

# ── Final summary ─────────────────────────────────────────────────────────────
if reward_history:
    print('\n=== FINAL REWARD SUMMARY ===')
    for role in ALL_ROLES:
        vals = reward_history.get(role, [])
        if not vals:
            continue
        n   = len(vals)
        avg = sum(vals) / n
        fq  = sum(vals[:max(1, n//4)]) / max(1, n//4)
        lq  = sum(vals[max(0, 3*n//4):]) / max(1, n - 3*n//4)
        arrow = '↑' if lq > fq + 0.03 else ('↓' if lq < fq - 0.03 else '→')
        adapt_tag = ' ◆' if role == 'ADAPT' else '  '
        print(f'  {role:12s}{adapt_tag}: mean={avg:+.3f}  first_q={fq:+.3f}  last_q={lq:+.3f}  {arrow}')

status = 'complete' if proc.returncode == 0 else f'FAILED (exit code {proc.returncode})'
print(f'\nTraining {status} in {elapsed/60:.1f} min')

## 9. Plot reward curves

All plots saved to `OUTPUT_DIR/plots/*.png`.

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from pathlib import Path

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
curves_path = OUTPUT_DIR / 'reward_curves.json'

if not curves_path.exists():
    print(f'reward_curves.json not found at {curves_path}')
else:
    with open(curves_path) as f:
        curves = json.load(f)

    def _ema(vals, span=15):
        if len(vals) < 2:
            return vals
        alpha, ema, out = 2 / (span + 1), vals[0], []
        for v in vals:
            ema = alpha * v + (1 - alpha) * ema
            out.append(ema)
        return out

    # ── Plot 1: Per-role reward curves (5 panels) ──────────────────────────────
    role_colors = {
        'AMAN':       '#2196F3',
        'DMAN':       '#4CAF50',
        'GENERATOR':  '#FF9800',
        'SUPERVISOR': '#9C27B0',
        'ADAPT':      '#F44336',  # red — domain-transfer agent
    }
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    ax_list = axes.flatten()
    for idx, (role, color) in enumerate(role_colors.items()):
        ax  = ax_list[idx]
        raw = curves.get(role, [])
        if not raw:
            ax.text(0.5, 0.5, f'No {role} data', ha='center', va='center',
                    transform=ax.transAxes, color='grey')
            ax.set_title(role)
            continue
        steps = range(len(raw))
        ax.plot(steps, raw,       alpha=0.25, color=color, lw=0.8, label='raw')
        ax.plot(steps, _ema(raw), color=color, lw=2.0,     label='EMA-15')
        ax.axhline(0, color='black', lw=0.5, ls='--')
        ax.set_xlabel('Training step')
        ax.set_ylabel('Reward')
        label_suffix = ' (domain transfer)' if role == 'ADAPT' else ''
        ax.set_title(f'{role}{label_suffix}', fontweight='bold')
        ax.set_ylim(-1.05, 1.05)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    # 6th panel: composite
    ax_comp = ax_list[5]
    comp    = curves.get('composite', [])
    if comp:
        steps = range(len(comp))
        ax_comp.plot(steps, comp,       alpha=0.25, color='#607D8B', lw=0.8, label='raw')
        ax_comp.plot(steps, _ema(comp), color='#607D8B', lw=2.0,     label='EMA-15')
        ax_comp.axhline(0, color='black', lw=0.5, ls='--')
        ax_comp.set_xlabel('Training step')
        ax_comp.set_ylabel('Reward')
        ax_comp.set_title('Composite (all roles)', fontweight='bold')
        ax_comp.set_ylim(-1.05, 1.05)
        ax_comp.legend(fontsize=9)
        ax_comp.grid(alpha=0.3)
    else:
        ax_list[5].set_visible(False)

    fig.suptitle('ATC Multi-Agent GRPO — Per-Role Reward Curves (incl. ADAPT)',
                 fontsize=14, fontweight='bold', y=1.01)
    fig.tight_layout()
    p1 = PLOTS_DIR / 'per_role_rewards.png'
    fig.savefig(p1, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {p1}')
    display(Image(filename=str(p1)))

    # ── Plot 2: Composite reward + rolling std (GRPO health) ──────────────────
    composite = curves.get('composite', [])
    if composite:
        w    = 20
        stds = [float(np.std(composite[max(0, i-w):i+1])) for i in range(len(composite))]
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        steps = range(len(composite))
        ax1.plot(steps, composite,       alpha=0.25, color='#607D8B', lw=0.8)
        ax1.plot(steps, _ema(composite), color='#607D8B', lw=2.0, label='EMA-15')
        ax1.axhline(0, color='black', lw=0.6, ls='--')
        ax1.set_ylabel('Composite reward')
        ax1.set_ylim(-1.05, 1.05)
        ax1.set_title('Composite reward (all 5 roles)', fontweight='bold')
        ax1.grid(alpha=0.3)
        ax1.legend(fontsize=9)
        ax2.plot(steps, stds, color='#E91E63', lw=1.5, label=f'rolling std (w={w})')
        ax2.axhline(0.01, color='red', lw=0.8, ls=':', label='collapse threshold 0.01')
        ax2.set_xlabel('Training step')
        ax2.set_ylabel('Within-group std')
        ax2.set_title('Reward diversity — must stay > 0.01 for GRPO gradient', fontweight='bold')
        ax2.grid(alpha=0.3)
        ax2.legend(fontsize=9)
        fig.tight_layout()
        p2 = PLOTS_DIR / 'composite_and_std.png'
        fig.savefig(p2, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'Saved: {p2}')
        display(Image(filename=str(p2)))

    # ── Plot 3: ADAPT domain-transfer deep dive ────────────────────────────────
    adapt_vals = curves.get('ADAPT', [])
    if len(adapt_vals) >= 4:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Left: ADAPT reward over training
        ax = axes[0]
        steps = range(len(adapt_vals))
        ax.plot(steps, adapt_vals,       alpha=0.25, color='#F44336', lw=0.8, label='raw')
        ax.plot(steps, _ema(adapt_vals), color='#F44336', lw=2.5,     label='EMA-15')
        ax.axhline(0, color='black', lw=0.6, ls='--')
        ax.set_xlabel('Training step')
        ax.set_ylabel('Reward')
        ax.set_title('ADAPT: domain-transfer reward over training', fontweight='bold')
        ax.set_ylim(-1.05, 1.05)
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

        # Right: ADAPT vs other roles first/last quarter comparison
        ax  = axes[1]
        all_roles_data = {r: curves.get(r, []) for r in
                          ['AMAN', 'DMAN', 'GENERATOR', 'SUPERVISOR', 'ADAPT']}
        roles_ok = [r for r, v in all_roles_data.items() if len(v) >= 8]
        x = np.arange(len(roles_ok))
        fq_vals, lq_vals, colors_bar = [], [], []
        _role_colors = {
            'AMAN': '#90CAF9', 'DMAN': '#A5D6A7',
            'GENERATOR': '#FFCC80', 'SUPERVISOR': '#CE93D8', 'ADAPT': '#EF9A9A',
        }
        for role in roles_ok:
            r = all_roles_data[role]
            n = len(r)
            fq_vals.append(float(np.mean(r[:max(1, n//4)])))
            lq_vals.append(float(np.mean(r[max(0, 3*n//4):])))
            colors_bar.append(_role_colors.get(role, '#BDBDBD'))
        b1 = ax.bar(x - 0.2, fq_vals, 0.35, label='First quarter',
                    color=[c + '88' for c in colors_bar] if False else colors_bar,
                    edgecolor='#333', alpha=0.6)
        b2 = ax.bar(x + 0.2, lq_vals, 0.35, label='Last quarter',
                    color=colors_bar, edgecolor='#333')
        for b in list(b1) + list(b2):
            ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                    f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(x)
        ax.set_xticklabels(roles_ok, fontsize=11)
        ax.set_ylabel('Mean reward')
        ax.set_title('First vs last quarter — evidence of learning per role', fontweight='bold')
        ax.axhline(0, color='black', lw=0.6, ls='--')
        ax.legend()
        ax.grid(axis='y', alpha=0.3)

        fig.suptitle('ADAPT Domain-Transfer Training Analysis', fontsize=13, fontweight='bold')
        fig.tight_layout()
        p3 = PLOTS_DIR / 'adapt_analysis.png'
        fig.savefig(p3, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'Saved: {p3}')
        display(Image(filename=str(p3)))

## 10. Before vs after comparison

Heuristic baseline (no LLM) · base model (untrained LoRA) · trained model  
Includes **ADAPT domain-transfer metrics** alongside ATC metrics.

In [ ]:
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from pathlib import Path

def _load(p):
    p = Path(p)
    return json.loads(p.read_text()) if p.exists() else None

baseline = _load(OUTPUT_DIR / 'baseline_metrics.json')
base_mdl = _load(OUTPUT_DIR / 'base_model_metrics.json')
trained  = _load(OUTPUT_DIR / 'trained_model_metrics.json')

if not any([baseline, base_mdl, trained]):
    print('No eval metrics found. Run training without --no_eval.')
else:
    keys = [
        ('mean_composite',        'Composite score'),
        ('mean_aman_reward',      'AMAN reward'),
        ('mean_dman_reward',      'DMAN reward'),
        ('mean_adapt_reward',     'ADAPT domain-transfer ◆'),
        ('mean_conflicts',        'Avg conflicts (↓ better)'),
        ('adapt_downstream_gain', 'ADAPT downstream gain ◆'),
    ]
    print(f'{"Metric":30s} {"Heuristic":>12} {"Base model":>12} {"Trained":>12}')
    print('-' * 70)
    for k, label in keys:
        bv = f'{baseline.get(k, float("nan")):.3f}' if baseline else 'N/A'
        mv = f'{base_mdl.get(k,  float("nan")):.3f}' if base_mdl else 'N/A'
        tv = f'{trained.get(k,   float("nan")):.3f}' if trained  else 'N/A'
        print(f'{label:30s} {bv:>12} {mv:>12} {tv:>12}')

    sources = []
    if baseline: sources.append(('Heuristic\nbaseline',      baseline, '#B0BEC5'))
    if base_mdl: sources.append(('Base model\n(no finetune)', base_mdl, '#90CAF9'))
    if trained:  sources.append(('Trained\nmodel',            trained,  '#A5D6A7'))

    if sources:
        # ATC metrics bar chart
        atc_keys = [
            ('mean_composite',    'Composite'),
            ('mean_aman_reward',  'AMAN'),
            ('mean_dman_reward',  'DMAN'),
        ]
        # ADAPT metrics bar chart
        adapt_keys = [
            ('mean_adapt_reward',     'ADAPT reward'),
            ('adapt_downstream_gain', 'Downstream gain'),
        ]

        fig, axes = plt.subplots(1, len(atc_keys) + len(adapt_keys),
                                  figsize=(5 * (len(atc_keys) + len(adapt_keys)), 5))
        all_plot_keys = atc_keys + adapt_keys
        is_adapt = [False] * len(atc_keys) + [True] * len(adapt_keys)

        for ax, (k, label), adapt_flag in zip(axes, all_plot_keys, is_adapt):
            vals   = [s[1].get(k, 0.0) for s in sources]
            labels = [s[0] for s in sources]
            colors = [s[2] for s in sources]
            if adapt_flag:
                colors = ['#FFCDD2', '#EF9A9A', '#F44336'][:len(sources)]
            bars = ax.bar(labels, vals, color=colors, edgecolor='#333', width=0.55)
            for b, v in zip(bars, vals):
                ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01,
                        f'{v:.3f}', ha='center', va='bottom', fontsize=10)
            title_suffix = ' ◆' if adapt_flag else ''
            ax.set_title(f'{label}{title_suffix}', fontweight='bold')
            ax.set_ylabel('Score')
            ax.set_ylim(0, 1.15)
            ax.grid(axis='y', alpha=0.3)
            if adapt_flag:
                ax.set_facecolor('#FFF8F8')

        fig.suptitle('Before vs After GRPO Training  (◆ = ADAPT domain-transfer metrics)',
                     fontsize=13, fontweight='bold', y=1.03)
        fig.tight_layout()
        p_eval = PLOTS_DIR / 'eval_comparison.png'
        fig.savefig(p_eval, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'\nSaved: {p_eval}')
        display(Image(filename=str(p_eval)))

## 11. Summary

In [ ]:
print('=' * 60)
print('  ATC + ADAPT GRPO Training — Run Complete')
print('=' * 60)
print(f'  Model:               {MODEL_NAME}')
print(f'  Episodes:            {EPISODES}')
print(f'  Group size:          {N_GENERATIONS}')
print(f'  LoRA rank:           {LORA_RANK}')
print(f'  Domain episode ratio:{DOMAIN_EPISODE_RATIO} → ADAPT agent')
print(f'  Output:              {OUTPUT_DIR}')
print(f'  Plots:               {PLOTS_DIR}')
print()
print('  Agents trained:')
print('    AMAN       — arrival sequencing & hold negotiation')
print('    DMAN       — departure ATFM compliance & prioritisation')
print('    GENERATOR  — adversarial scenario mutation')
print('    SUPERVISOR — preference-aware plan evaluation')
print('    ADAPT ◆    — domain-agnostic structural inference (ICU → ATC wake/priority)')
print()
print('  ADAPT reward composition:')
print('    0.70 × downstream AMAN+DMAN score on mapped task')
print('    0.15 × improvement over heuristic baseline on unmapped task')
print('    0.10 × entity coverage (all entity types mapped)')
print('    0.05 × rationale quality (verifiable structural reasoning)')
print()
print('  Output files:')
all_files = sorted(OUTPUT_DIR.rglob('*.json')) + sorted(PLOTS_DIR.rglob('*.png'))
for f in all_files:
    print(f'    {f.relative_to(OUTPUT_DIR)}')
print('=' * 60)

## Troubleshooting

| Symptom | Fix |
|---|---|
| `retry` ImportError | Re-run Cell 3 — do NOT add `--no-deps` to the unsloth install line |
| CUDA OOM | Set `N_GENERATIONS=2`; use `Qwen2.5-1.5B-Instruct` |
| `GRPOConfig` keyword error | Confirm `trl==0.15.2` printed in Cell 3 output |
| ADAPT role missing from dataset | Confirm `domain_episode_ratio > 0` and ICU domain loaded |
| ADAPT rewards constant | `os.environ['ATC_REWARD_TRACE']='1'` before training to trace per-component values |
| Rewards still constant (any role) | `os.environ['ATC_REWARD_TRACE']='1'` before training |
| Disconnect mid-training | Enable Drive mount (`USE_DRIVE=True`) |
| Kaggle: slow training | Keep `BATCH_SIZE=2`, `N_GENERATIONS=4` as-is |

**Reward trace**: `os.environ['ATC_REWARD_TRACE'] = '1'` — prints per-component breakdown per completion for all 5 roles.